In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/Colab Notebooks/NFV3DATA-A11964_A11964.zip"
extract_path = "/content/unzipped_data"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction completed successfully.")


Extraction completed successfully.


In [ ]:
import pandas as pd


# csv_path = "/content/unzipped_data/f7546561558c07c5_NFV3DATA-A11964_A11964/data/NF-UNSW-NB15-v3.csv"
csv_path = "/content/drive/MyDrive/Colab Notebooks/NF-ToN-IoT-v3.csv"
df = pd.read_csv(csv_path)
df.head()

In [ ]:
print(df.columns.tolist())

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import random

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

csv_path = "/content/drive/MyDrive/Colab Notebooks/NF-ToN-IoT-v3.csv"
df = pd.read_csv(csv_path, low_memory=False)

print("Dataset Shape:", df.shape)

# selected_features = [
#     "IN_BYTES", "OUT_BYTES",
#     "IN_PKTS", "OUT_PKTS",
#     "FLOW_DURATION_MILLISECONDS",
#     "LONGEST_FLOW_PKT", "SHORTEST_FLOW_PKT",
#     "TCP_FLAGS", "CLIENT_TCP_FLAGS", "SERVER_TCP_FLAGS",
#     "RETRANSMITTED_IN_PKTS",
#     "RETRANSMITTED_OUT_PKTS",
#     "SRC_TO_DST_IAT_AVG",
#     "SRC_TO_DST_IAT_STDDEV",
#     "DST_TO_SRC_IAT_AVG",
#     "DST_TO_SRC_IAT_STDDEV",
#     "SRC_TO_DST_AVG_THROUGHPUT",
#     "DST_TO_SRC_AVG_THROUGHPUT",
#     "MIN_TTL", "MAX_TTL",
#     "PROTOCOL", "L7_PROTO"
# ]

selected_features = [
    "IN_BYTES", "OUT_BYTES",
    "IN_PKTS", "OUT_PKTS",
    "FLOW_DURATION_MILLISECONDS",
    "LONGEST_FLOW_PKT", "SHORTEST_FLOW_PKT",
    "TCP_FLAGS", "CLIENT_TCP_FLAGS", "SERVER_TCP_FLAGS",
    "RETRANSMITTED_IN_PKTS",
    "RETRANSMITTED_OUT_PKTS",
    "MIN_TTL", "MAX_TTL",
    "PROTOCOL", "L7_PROTO"
]

df = df[selected_features + ["Label", "FLOW_START_MILLISECONDS"]]
df = df.sort_values("FLOW_START_MILLISECONDS")
split_index = int(len(df) * 0.8)
train_df = df.iloc[:split_index]
test_df  = df.iloc[split_index:]

y_train = train_df["Label"]
y_test  = test_df["Label"]

X_train = train_df[selected_features]
X_test  = test_df[selected_features]

#y_train = np.random.permutation(y_train) #leakage test

X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
X_test  = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

X_train = X_train.astype(np.float32)
X_test  = X_test.astype(np.float32)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train).astype(np.float32)
X_test  = scaler.transform(X_test).astype(np.float32)

# #-----
# from sklearn.linear_model import LogisticRegression

# print("\n=======================================")
# print("LOGISTIC REGRESSION BASELINE")
# print("=======================================")

# log_reg = LogisticRegression(
#     max_iter=1000,
#     class_weight="balanced",
#     n_jobs=-1
# )

# log_reg.fit(X_train, y_train)

# y_prob_lr = log_reg.predict_proba(X_test)[:, 1]
# y_pred_lr = (y_prob_lr > 0.5).astype(int)

# accuracy_lr = accuracy_score(y_test, y_pred_lr)
# precision_lr = precision_score(y_test, y_pred_lr)
# recall_lr = recall_score(y_test, y_pred_lr)
# f1_lr = f1_score(y_test, y_pred_lr)
# roc_auc_lr = roc_auc_score(y_test, y_prob_lr)

# print(f"Accuracy      : {accuracy_lr:.6f}")
# print(f"Precision     : {precision_lr:.6f}")
# print(f"Recall        : {recall_lr:.6f}")
# print(f"F1-Score      : {f1_lr:.6f}")
# print(f"ROC-AUC Score : {roc_auc_lr:.6f}")

# print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))
# print("\nClassification Report:\n", classification_report(y_test, y_pred_lr))
# print("===============\n")

# #----

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weight_dict = {
    0: class_weights[0],
    1: class_weights[1]
}

print("Class Weights:", class_weight_dict)

model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),

    layers.Dense(64, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(32, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(16, activation="relu"),
    layers.Dropout(0.2),

    layers.Dense(1, activation="sigmoid")
])

from tensorflow.keras import metrics

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
     metrics=[
        metrics.Precision(name="precision"),
        metrics.Recall(name="recall"),
        metrics.AUC(name="roc_auc"),
        metrics.AUC(name="pr_auc", curve="PR")
    ]
)

model.summary()

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=4096,
    class_weight=class_weight_dict,
    callbacks=[early_stop],
    verbose=1
)

y_prob = model.predict(X_test).ravel()
y_pred = (y_prob > 0.5).astype(int)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("\n=======================================")
print("MLP PERFORMANCE")
print("=======================================")
print(f"Accuracy      : {accuracy:.6f}")
print(f"Precision     : {precision:.6f}")
print(f"Recall        : {recall:.6f}")
print(f"F1-Score      : {f1:.6f}")
print(f"ROC-AUC Score : {roc_auc:.6f}")
print("=======================================\n")

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Dataset Shape: (2365424, 55)
Class Weights: {0: np.float64(0.5238923544244363), 1: np.float64(10.96359833605636)}


Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_28 (Dense)                │ (None, 64)             │         1,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_14          │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_21 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_29 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_15          │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_22 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_30 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_23 (Dropout)            │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_31 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,097 (16.00 KB)

 Trainable params: 3,905 (15.25 KB)

 Non-trainable params: 192 (768.00 B)

Epoch 1/20
370/370 ━━━━━━━━━━━━━━━━━━━━ 17s 28ms/step - loss: 0.2473 - pr_auc: 0.8348 - precision: 0.1984 - recall: 0.9671 - roc_auc: 0.9654 - val_loss: 0.0068 - val_pr_auc: 0.9989 - val_precision: 0.9960 - val_recall: 0.9978 - val_roc_auc: 0.9998
Epoch 2/20
370/370 ━━━━━━━━━━━━━━━━━━━━ 9s 24ms/step - loss: 0.0147 - pr_auc: 0.9929 - precision: 0.9467 - recall: 0.9981 - roc_auc: 0.9996 - val_loss: 0.0044 - val_pr_auc: 0.9960 - val_precision: 0.9949 - val_recall: 0.9994 - val_roc_auc: 0.9996
Epoch 3/20
370/370 ━━━━━━━━━━━━━━━━━━━━ 10s 27ms/step - loss: 0.0077 - pr_auc: 0.9942 - precision: 0.9797 - recall: 0.9987 - roc_auc: 0.9997 - val_loss: 0.0050 - val_pr_auc: 0.9961 - val_precision: 0.9937 - val_recall: 0.9995 - val_roc_auc: 0.9997
Epoch 4/20
370/370 ━━━━━━━━━━━━━━━━━━━━ 9s 23ms/step - loss: 0.0057 - pr_auc: 0.9948 - precision: 0.9842 - recall: 0.9992 - roc_auc: 0.9997 - val_loss: 0.0040 - val_pr_auc: 0.9990 - val_precision: 0.9937 - val_recall: 0.9996 - val_roc_auc: 0.9998
Epoch 5/20

In [ ]:
# #leakage safe

# import os, random
# import numpy as np
# import pandas as pd
# import tensorflow as tf

# SEED = 42
# os.environ["PYTHONHASHSEED"] = str(SEED)
# np.random.seed(SEED)
# random.seed(SEED)
# tf.random.set_seed(SEED)

# from tensorflow import keras
# from tensorflow.keras import layers
# from sklearn.preprocessing import StandardScaler
# from sklearn.metrics import roc_auc_score

# csv_path = "/content/unzipped_data/f7546561558c07c5_NFV3DATA-A11964_A11964/data/NF-UNSW-NB15-v3.csv"
# df = pd.read_csv(csv_path, low_memory=False)

# print("Original dataset size:", len(df))

# selected_features = [
#     "LONGEST_FLOW_PKT",
#     "SHORTEST_FLOW_PKT",
#     "TCP_FLAGS",
#     "CLIENT_TCP_FLAGS",
#     "SERVER_TCP_FLAGS",
#     "RETRANSMITTED_IN_PKTS",
#     "RETRANSMITTED_OUT_PKTS",
#     "SRC_TO_DST_IAT_STDDEV",
#     "DST_TO_SRC_IAT_STDDEV",
#     "MIN_TTL",
#     "MAX_TTL"
# ]

# df = df[selected_features + ["Label", "FLOW_START_MILLISECONDS"]]
# df_unique = df.drop_duplicates(subset=selected_features)
# print("After duplicate removal:", len(df_unique))

# df_unique = df_unique.sort_values("FLOW_START_MILLISECONDS")

# n = len(df_unique)
# train_end = int(0.7 * n)
# val_end   = int(0.8 * n)

# train_df = df_unique.iloc[:train_end]
# val_df   = df_unique.iloc[train_end:val_end]
# test_df  = df_unique.iloc[val_end:]
# test_df = test_df.merge(
#     train_df[selected_features],
#     on=selected_features,
#     how="left",
#     indicator=True
# )

# test_df = test_df[test_df["_merge"] == "left_only"]
# test_df = test_df.drop(columns=["_merge"])

# print("Final test size:", len(test_df))

# cross_check = pd.merge(
#     train_df[selected_features],
#     test_df[selected_features],
#     how="inner"
# )

# print("Cross-split duplicates:", len(cross_check))
# X_train = train_df[selected_features].copy()
# y_train = train_df["Label"].copy()

# X_val   = val_df[selected_features].copy()
# y_val   = val_df["Label"].copy()

# X_test  = test_df[selected_features].copy()
# y_test  = test_df["Label"].copy()

# for X in [X_train, X_val, X_test]:
#     X.replace([np.inf, -np.inf], np.nan, inplace=True)
#     X.fillna(0, inplace=True)

# scaler = StandardScaler()
# X_train = scaler.fit_transform(X_train)
# X_val   = scaler.transform(X_val)
# X_test  = scaler.transform(X_test)


# # ============================================================
# # 8️⃣ BUILD MODEL
# # ============================================================

# def build_model(input_dim):
#     model = keras.Sequential([
#         layers.Input(shape=(input_dim,)),
#         layers.Dense(128, activation="relu"),
#         layers.BatchNormalization(),
#         layers.Dense(64, activation="relu"),
#         layers.BatchNormalization(),
#         layers.Dense(32, activation="relu"),
#         layers.Dense(1, activation="sigmoid")
#     ])
#     model.compile(
#         optimizer="adam",
#         loss="binary_crossentropy",
#         metrics=[tf.keras.metrics.AUC(name="auc")]
#     )
#     return model
# model = build_model(X_train.shape[1])

# early_stop = keras.callbacks.EarlyStopping(
#     monitor="val_auc",
#     mode="max",
#     patience=5,
#     restore_best_weights=True
# )

# model.fit(
#     X_train,
#     y_train,
#     validation_data=(X_val, y_val),
#     epochs=30,
#     batch_size=256,
#     callbacks=[early_stop],
#     verbose=1
# )

# y_prob = model.predict(X_test).ravel()
# roc = roc_auc_score(y_test, y_prob)

# print("\nFinal Test ROC-AUC:", roc)

# print("\nRunning label shuffle test...")

# tf.keras.backend.clear_session()

# y_train_shuffled = np.random.permutation(y_train)

# model_shuffle = build_model(X_train.shape[1])

# model_shuffle.fit(
#     X_train,
#     y_train_shuffled,
#     epochs=5,
#     batch_size=256,
#     verbose=0
# )

# y_prob_shuffle = model_shuffle.predict(X_test).ravel()
# roc_shuffle = roc_auc_score(y_test, y_prob_shuffle)

# print("ROC-AUC after label shuffle:", roc_shuffle)
# print("Expected ≈ 0.50")

Original dataset size: 2365424
After duplicate removal: 249508
Final test size: 49902
Cross-split duplicates: 0
Epoch 1/30
683/683 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - auc: 0.9933 - loss: 0.0709 - val_auc: 0.9997 - val_loss: 0.0056
Epoch 2/30
683/683 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - auc: 0.9995 - loss: 0.0070 - val_auc: 0.9998 - val_loss: 0.0047
Epoch 3/30
683/683 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - auc: 0.9997 - loss: 0.0055 - val_auc: 0.9998 - val_loss: 0.0056
Epoch 4/30
683/683 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - auc: 0.9998 - loss: 0.0050 - val_auc: 0.9997 - val_loss: 0.0074
Epoch 5/30
683/683 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - auc: 0.9998 - loss: 0.0043 - val_auc: 0.9997 - val_loss: 0.0052
Epoch 6/30
683/683 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - auc: 0.9999 - loss: 0.0038 - val_auc: 0.9999 - val_loss: 0.0041
Epoch 7/30
683/683 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - auc: 0.9999 - loss: 0.0034 - val_auc: 0.9995 - val_loss: 0.0067
Epoch 8/30
683/683 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - auc: 0.9

In [ ]:
# #structural-leakage safe

# import os, random
# import numpy as np
# import pandas as pd
# import tensorflow as tf

# SEED = 42
# os.environ["PYTHONHASHSEED"] = str(SEED)
# np.random.seed(SEED)
# random.seed(SEED)
# tf.random.set_seed(SEED)

# from tensorflow import keras
# from tensorflow.keras import layers
# from sklearn.preprocessing import StandardScaler
# from sklearn.cluster import MiniBatchKMeans
# from sklearn.metrics import roc_auc_score

# csv_path = "/content/unzipped_data/f7546561558c07c5_NFV3DATA-A11964_A11964/data/NF-UNSW-NB15-v3.csv"
# df = pd.read_csv(csv_path, low_memory=False)

# print("Original size:", len(df))

# selected_features = [
#     "LONGEST_FLOW_PKT",
#     "SHORTEST_FLOW_PKT",
#     "TCP_FLAGS",
#     "CLIENT_TCP_FLAGS",
#     "SERVER_TCP_FLAGS",
#     "RETRANSMITTED_IN_PKTS",
#     "RETRANSMITTED_OUT_PKTS",
#     "SRC_TO_DST_IAT_STDDEV",
#     "DST_TO_SRC_IAT_STDDEV",
#     "MIN_TTL",
#     "MAX_TTL"
# ]

# df = df[selected_features + ["Label", "FLOW_START_MILLISECONDS"]]

# df = df.drop_duplicates(subset=selected_features)
# print("After deduplication:", len(df))
# print("Clustering to remove structural redundancy...")

# scaler_cluster = StandardScaler()
# X_cluster = scaler_cluster.fit_transform(df[selected_features])

# kmeans = MiniBatchKMeans(
#     n_clusters=5000,
#     batch_size=10000,
#     random_state=SEED
# )

# clusters = kmeans.fit_predict(X_cluster)

# df["cluster"] = clusters
# df_reduced = df.groupby("cluster").first().reset_index(drop=True)

# print("After cluster reduction:", len(df_reduced))

# df_reduced = df_reduced.sort_values("FLOW_START_MILLISECONDS")

# split_idx = int(0.5 * len(df_reduced))

# train_df = df_reduced.iloc[:split_idx]
# test_df  = df_reduced.iloc[split_idx:]

# print("Train size:", len(train_df))
# print("Test size :", len(test_df))

# X_train = train_df[selected_features].copy()
# y_train = train_df["Label"].copy()

# X_test  = test_df[selected_features].copy()
# y_test  = test_df["Label"].copy()

# for X in [X_train, X_test]:
#     X.replace([np.inf, -np.inf], np.nan, inplace=True)
#     X.fillna(0, inplace=True)

# scaler = StandardScaler()
# X_train = scaler.fit_transform(X_train)
# X_test  = scaler.transform(X_test)


# def build_model(input_dim):
#     model = keras.Sequential([
#         layers.Input(shape=(input_dim,)),
#         layers.Dense(128, activation="relu"),
#         layers.BatchNormalization(),
#         layers.Dense(64, activation="relu"),
#         layers.BatchNormalization(),
#         layers.Dense(32, activation="relu"),
#         layers.Dense(1, activation="sigmoid")
#     ])
#     model.compile(
#         optimizer="adam",
#         loss="binary_crossentropy",
#         metrics=[tf.keras.metrics.AUC(name="auc")]
#     )
#     return model


# model = build_model(X_train.shape[1])

# model.fit(
#     X_train,
#     y_train,
#     epochs=30,
#     batch_size=256,
#     verbose=1
# )

# y_prob = model.predict(X_test).ravel()
# roc = roc_auc_score(y_test, y_prob)

# print("\nFINAL ROC-AUC:", roc)

# print("\nRunning TRUE global shuffle test...")

# df_shuffle = df_reduced.copy()
# df_shuffle["Label"] = np.random.permutation(df_shuffle["Label"].values)

# df_shuffle = df_shuffle.sort_values("FLOW_START_MILLISECONDS")

# train_s = df_shuffle.iloc[:split_idx]
# test_s  = df_shuffle.iloc[split_idx:]

# X_train_s = scaler.fit_transform(train_s[selected_features])
# X_test_s  = scaler.transform(test_s[selected_features])

# y_train_s = train_s["Label"]
# y_test_s  = test_s["Label"]

# tf.keras.backend.clear_session()

# model_s = build_model(X_train_s.shape[1])

# model_s.fit(
#     X_train_s,
#     y_train_s,
#     epochs=5,
#     batch_size=256,
#     verbose=0
# )

# y_prob_s = model_s.predict(X_test_s).ravel()
# roc_s = roc_auc_score(y_test_s, y_prob_s)

# print("Shuffle ROC-AUC:", roc_s)
# print("Expected ≈ 0.5")

Original size: 2365424
After deduplication: 249508
Clustering to remove structural redundancy...
After cluster reduction: 4997
Train size: 2498
Test size : 2499
Epoch 1/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - auc: 0.7394 - loss: 0.6950
Epoch 2/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - auc: 0.9920 - loss: 0.1770
Epoch 3/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - auc: 0.9974 - loss: 0.1017
Epoch 4/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - auc: 0.9984 - loss: 0.0665
Epoch 5/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - auc: 0.9987 - loss: 0.0465
Epoch 6/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - auc: 0.9990 - loss: 0.0328
Epoch 7/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - auc: 0.9993 - loss: 0.0248
Epoch 8/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - auc: 0.9995 - loss: 0.0199
Epoch 9/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - auc: 0.9995 - loss: 0.0167
Epoch 10/30
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - auc: 0.9996 - loss: 0.0144
Epoch 11/30
10/10 ━━━━━━━━━━━━━━━━━━━━